In [108]:
import pandas as pd
import ijson
import matplotlib.pyplot as plt
import seaborn as sns
import re
import random

random.seed(2)

In [109]:
with open("train.json", "r", encoding="utf-8") as f:
    data = []
    for item in ijson.items(f, "item"):
        data.append(item)

random.shuffle(data)
data = data[:10000]

train = pd.DataFrame(data)

with open("test.json", "r", encoding="utf-8") as f:
    data = []
    for i, item in enumerate(ijson.items(f, "item")):
        data.append(item)
        
random.shuffle(data)
data = data[:10000]

test = pd.DataFrame(data)


with open("validation.json", "r", encoding="utf-8") as f:
    data = []
    for i, item in enumerate(ijson.items(f, "item")):
        data.append(item)

    random.shuffle(data)
    data = data[:10000]

validation = pd.DataFrame(data)

In [110]:
train = train[train['label'] != 'neutral']
validation = validation[validation['label'] != 'neutral']
test = test[test['label'] != 'neutral']

In [112]:
import fasttext
import plotly.express as px

# Load the pre-trained model
model = fasttext.load_model('lid.176.bin')

def is_english(text, threshold=0.8):
    # Remove newlines for better detection
    text = text.replace("\n", " ")
    prediction = model.predict(text, k=1)
    
    lang = prediction[0][0].replace('__label__', '')
    confidence = prediction[1][0]
    
    return lang == 'en' and confidence >= threshold

mask_train = train['text'].apply(lambda x: is_english(x, threshold=0.9))
train = train[mask_train]

mask_validation = validation['text'].apply(lambda x: is_english(x, threshold=0.9))
validation = validation[mask_validation]

mask_test = test['text'].apply(lambda x: is_english(x, threshold=0.9))
test = test[mask_test]

In [114]:
import re

def procesar_hashtags(text):
    
    def separar_hashtag(match):
        hashtag = match.group()[1:]  # quitar #

        # separar CamelCase
        hashtag = re.sub(r'([a-z])([A-Z])', r'\1 \2', hashtag)

        return hashtag

    return re.sub(r'#\w+', separar_hashtag, text)

train['text'] = train['text'].apply(procesar_hashtags)
validation['text'] = validation['text'].apply(procesar_hashtags)
test['text'] = test['text'].apply(procesar_hashtags)


In [116]:
import re

def eliminar_urls(text):
    return re.sub(r'http\S+|www\S+|https\S+', '', text)

train['text'] = train['text'].apply(eliminar_urls)
validation['text'] = validation['text'].apply(eliminar_urls)
test['text'] = test['text'].apply(eliminar_urls)

In [ ]:
import re

def eliminar_menciones(text):
    return re.sub(r'@\w+', '', text)

train['text'] = train['text'].apply(eliminar_menciones)
validation['text'] = validation['text'].apply(eliminar_menciones)
test['text'] = test['text'].apply(eliminar_menciones)

In [118]:
import emoji

# Función robusta para contar emojis reales
def contar_emojis_reales(text):
    if not isinstance(text, str):
        return 0
    # emoji.emoji_count cuenta específicamente pictogramas Unicode de la base de datos oficial
    return emoji.emoji_count(text)

train['emoji_count'] = train['text'].apply(contar_emojis_reales)
validation['emoji_count'] = validation['text'].apply(contar_emojis_reales)
test['emoji_count'] = test['text'].apply(contar_emojis_reales)

train["has_emojis"] = train['emoji_count'] > 0
validation["has_emojis"] = validation['emoji_count'] > 0
test["has_emojis"] = test['emoji_count'] > 0

train = train.drop(columns=['emoji_count'])
validation = validation.drop(columns=['emoji_count'])
test = test.drop(columns=['emoji_count'])

train = train[train['has_emojis'] != 1]
validation = validation[validation['has_emojis'] != 1]
test = test[test['has_emojis'] != 1]

In [120]:
train = train.drop_duplicates(subset=['text'])
validation = validation.drop_duplicates(subset=['text'])
test = test.drop_duplicates(subset=['text'])

In [122]:
# Unir todos los datasets
joined = pd.concat(
    [
        train.assign(dataset="train"),
        validation.assign(dataset="validation"),
        test.assign(dataset="test")
    ],
    ignore_index=True
)

# Eliminar duplicados globales por texto
joined = joined.drop_duplicates(subset=["text"]).reset_index(drop=True)

# Separar otra vez
train = joined[joined["dataset"] == "train"].drop(columns="dataset").reset_index(drop=True)

validation = joined[joined["dataset"] == "validation"].drop(columns="dataset").reset_index(drop=True)

test = joined[joined["dataset"] == "test"].drop(columns="dataset").reset_index(drop=True)

In [124]:
import plotly.express as px

# TRAIN
fig = px.histogram(
    train,
    x='label',
    title='Distribution of Classes in Train',
    text_auto=True
)

fig.update_traces(textposition='inside')
fig.show()

print(train['label'].value_counts(normalize=True))


# VALIDATION
fig = px.histogram(
    validation,
    x='label',
    title='Distribution of Classes in Validation',
    text_auto=True
)

fig.update_traces(textposition='inside')
fig.show()

print(validation['label'].value_counts(normalize=True))


# TEST
fig = px.histogram(
    test,
    x='label',
    title='Distribution of Classes in Test',
    text_auto=True
)

fig.update_traces(textposition='inside')
fig.show()

print(test['label'].value_counts(normalize=True))

label
negative    0.555272
positive    0.444728
Name: proportion, dtype: float64


label
positive    0.500434
negative    0.499566
Name: proportion, dtype: float64


label
negative    0.563871
positive    0.436129
Name: proportion, dtype: float64


In [125]:
train.drop(columns=['has_emojis'], inplace=True)
validation.drop(columns=['has_emojis'], inplace=True)
test.drop(columns=['has_emojis'], inplace=True)

In [126]:
train.to_csv("train_preprocessed.csv", index=False)
validation.to_csv("validation_preprocessed.csv", index=False)
test.to_csv("test_preprocessed.csv", index=False)